<a href="https://colab.research.google.com/github/EdvanylsonAssuncao/curso_IA_generativa/blob/main/rede_neural_artificial_e_deep_learning_limite_credito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Importando as bibliotecas
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [ ]:
# 2. Carregando os dados
df = pd.read_csv('limites_de_credito_1000_registros.csv')
print("Visualizando os primeiros registros da base:\n")
print(df.head())

Visualizando os primeiros registros da base:

   Idade  Salario  Score_Credito  Limite_Aprovado
0     68     5133            533          2797.23
1     27     4621            706          4658.81
2     30    22716            786         11425.50
3     68    15653            940         10384.01
4     24     4957            433          2055.79


In [ ]:
# 3. Separando entradas e saídas
X = df[['Idade', 'Salario', 'Score_Credito']]
y = df['Limite_Aprovado']

In [ ]:
# 4. Separando treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    )

print("Tamanho do conjunto de treino:", X_treino.shape)
print("Tamanho do conjnto de teste:", X_teste.shape)

Tamanho do conjunto de treino: (800, 3)
Tamanho do conjnto de teste: (200, 3)


In [ ]:
# 5. Normalizando as entradas e o alvo

normalizador_x = StandardScaler()
X_treino_norm = normalizador_x.fit_transform(X_treino)
X_teste_norm = normalizador_x.transform(X_teste)

normalizador_y = StandardScaler()
y_treino_norm = normalizador_y.fit_transform(y_treino.values.reshape(-1, 1))
y_teste_norm = normalizador_y.transform(y_teste.values.reshape(-1, 1))

print("Normalização comcluida!")

Normalização comcluida!


In [ ]:
# 6. Construíndo a Rede Neural de regressão

tf.random.set_seed(42)

modelo_limite = keras.Sequential([
    keras.Input(shape=(3,)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(1)
])

modelo_limite.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

modelo_limite.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 16)             │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 209 (836.00 B)

 Trainable params: 209 (836.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 7. Treinando o modelo

print("Treinando o avaliador de limites ...\n")

historico = modelo_limite.fit(
    X_treino_norm,
    y_treino_norm,
    epochs=100,
    validation_split=0.2,
    verbose=1
)

print("\nTreinamento concluído!")


Treinando o avaliador de limites ...

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 1.8949 - mae: 1.1579 - val_loss: 1.5814 - val_mae: 1.0561
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.4693 - mae: 1.0136 - val_loss: 1.2298 - val_mae: 0.9258
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.1599 - mae: 0.8959 - val_loss: 0.9782 - val_mae: 0.8194
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.9104 - mae: 0.7880 - val_loss: 0.7738 - val_mae: 0.7171
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.7020 - mae: 0.6823 - val_loss: 0.6116 - val_mae: 0.6214
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.5447 - mae: 0.5880 - val_loss: 0.4865 - val_mae: 0.5416
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4277 - mae: 0.5086 - val_loss: 0.3898 - val_mae: 0.4732
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3383 - mae: 0.4425 - val_loss: 0.3168 - val_mae: 0.4182
Epoch 9/100
20/20 ━━━━━━━

In [ ]:
# 8. Avaliando modelo

perda, mae = modelo_limite.evaluate(X_teste_norm, y_teste_norm)

print(f"Perda no teste, e escala normalizada: {perda:.4f}")
print(f"MAE no teste, em escala normalizada: {mae:.4f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0699 - mae: 0.2132  
Perda no teste, e escala normalizada: 0.0699
MAE no teste, em escala normalizada: 0.2132


In [ ]:
# 9. Convertendo previsões para Reais

previsoes_norm = modelo_limite.predict(X_teste_norm)
previsoes_reais = normalizador_y.inverse_transform(previsoes_norm)
y_teste_reais = y_teste.values.reshape(-1, 1)
mae_reais = mean_absolute_error(y_teste_reais, previsoes_reais)
rmse_reais = np.sqrt(mean_squared_error(y_teste_reais, previsoes_reais))

print(f"Erro médio absoluto em Reais: R$ {mae_reais:.2f}")
print(f"Raiz do erro quadrántico médio em Reais: R$ {rmse_reais:.2f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
Erro médio absoluto em Reais: R$ 576.21
Raiz do erro quadrántico médio em Reais: R$ 714.46


In [ ]:
# 10. Testando com um novo cliente

cliente_novo = pd.DataFrame({
        'Idade': [35],
        'Salario': [8000],
        'Score_Credito': [750]
})

cliente_norm = normalizador_x.transform(cliente_novo)
limite_previsto_norm = modelo_limite.predict(cliente_norm)
limite_real = normalizador_y.inverse_transform(limite_previsto_norm)

print(f"O limite de crédito sugerido pela IA é: {limite_real[0][0]:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
O limite de crédito sugerido pela IA é: 5469.97
